# Botanix — PlantCLEF Pre-train → Fine-tune (CNN)
**PlantCLEF 2026:** https://www.imageclef.org/PlantCLEF2026  
**GPU:** RTX 5090 (Vast.ai)

## Çalışma Akışı (3 Faz)

| Faz | Dataset | Görev | Açıklama |
|-----|---------|-------|---------|
| **Faz 1** | PlantCLEF (~1.4M görüntü, 7.806 tür) | Single-label | Modeli büyük botanik veri setiyle pre-train et |
| **Faz 2** | Kendi hastalık veri seti (105 sınıf) | Single-label | Fine-tune: classification head sıfırla, backbone dondur/açık |
| **Faz 3** | PlantCLEF plot test seti | Multi-label | Vejetasyon plot'larında çoklu tür tespiti (isteğe bağlı) |

> **Not:** Model 1–5 karşılaştırma sonuçlarına göre en iyi mimari belirlendikten sonra  
> bu notebook çalıştırılmalıdır. Faz 1 ve 2 sırayla çalıştırılır.

In [ ]:
# ── FAZ 1: PlantCLEF Dataset İndirme ─────────────────────────────────────────
# PlantCLEF 2026, PlantCLEF 2024/2025 eğitim verisini kullanmaktadır.
# Kaggle yarışma sayfasından indirme: https://www.kaggle.com/competitions/plantclef-2026
# Doğrudan dosya indirme (~281GB veya ~160GB küçük versiyon):
#   https://lab.plantnet.org/LifeCLEF/PlantCLEF2024/single_plant_training_data/
#   PlantCLEF2024singleplanttrainingdata_800_max_side_size.tar  (~160GB önerilir)
#
# Vast.ai'de terminal ile indirme:
#   wget -c "https://lab.plantnet.org/LifeCLEF/PlantCLEF2024/single_plant_training_data/PlantCLEF2024singleplanttrainingdata_800_max_side_size.tar" -P ../plantclef/
#   tar -xf ../plantclef/PlantCLEF2024singleplanttrainingdata_800_max_side_size.tar -C ../plantclef/
#
# Metadata (sınıf etiketleri için gerekli):
#   wget "https://lab.plantnet.org/LifeCLEF/PlantCLEF2024/single_plant_training_data/PlantCLEF2024singleplanttrainingdata.csv" -P ../plantclef/

!pip install -q kagglehub pandas

import os, shutil
from pathlib import Path

PLANTCLEF_DIR = Path("../plantclef")
FINETUNE_DIR  = Path("../data")          # Kendi hastalık veri setin

print(f"PlantCLEF dizini : {PLANTCLEF_DIR.resolve()}")
print(f"Fine-tune dizini : {FINETUNE_DIR.resolve()}")
print(f"PlantCLEF mevcut: {PLANTCLEF_DIR.exists()}")
print(f"Fine-tune mevcut: {FINETUNE_DIR.exists()}")

# ── Kendi hastalık veri setini indir (fine-tune için) ────────────────────────
import kagglehub
_raw = kagglehub.dataset_download("mertcangelbal/plant-leaf-disease-classification-dataset")
print("\nHastalık dataset cache yolu:", _raw)
if not FINETUNE_DIR.exists():
    shutil.copytree(_raw, str(FINETUNE_DIR))
    print(f"Fine-tune dataset kopyalandı: {FINETUNE_DIR}")
else:
    print(f"Fine-tune dataset zaten mevcut: {FINETUNE_DIR}")

In [ ]:
# ── Kütüphaneler ─────────────────────────────────────────────────────────────
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print(f"PyTorch: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Konfigürasyon ─────────────────────────────────────────────────────────────
# ── Konfigürasyon ─────────────────────────────────────────────────────────────
CONFIG = {
    # Dizinler
    "plantclef_dir":     "../plantclef",       # PlantCLEF eğitim verisi
    "plantclef_meta":    "../plantclef/PlantCLEF2024singleplanttrainingdata.csv",
    "finetune_dir":      "../data",             # Kendi hastalık veri setin

    # Görüntü
    "img_size":          224,
    "batch_size":        32,
    "num_workers":       4,

    # Faz 1 — PlantCLEF Pre-train
    "plantclef_classes": 7806,                  # PlantCLEF 2024 tür sayısı
    "pretrain_epochs":   30,
    "pretrain_lr":       3e-4,
    "pretrain_save":     "./checkpoints/botanix_pretrained.pth",

    # Faz 2 — Fine-tune
    "finetune_classes":  105,                   # Kendi veri setindeki sınıf sayısı
    "finetune_epochs":   25,
    "finetune_lr":       1e-4,                  # Pre-train'den daha düşük LR
    "freeze_backbone":   True,                  # İlk epoch'larda backbone dondur
    "freeze_epochs":     5,                     # 5 epoch donuk, sonra aç
    "finetune_save":     "./checkpoints/botanix_finetuned_best.pth",

    "weight_decay":      1e-4,
    "model_name":        "botanix_cnn",
}
os.makedirs("./checkpoints", exist_ok=True)
os.makedirs("./results", exist_ok=True)
print("Konfigürasyon yüklendi.")
print(f"PlantCLEF: {CONFIG['plantclef_classes']} sınıf | Fine-tune: {CONFIG['finetune_classes']} sınıf")

In [ ]:
# ── Veri Dönüşümleri ──────────────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# PlantCLEF Faz 1 — agresif augmentation (scratch eğitim)
pretrain_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"] + 32, CONFIG["img_size"] + 32)),
    transforms.RandomCrop(CONFIG["img_size"]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Fine-tune Faz 2 — orta augmentation (domain shift)
finetune_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
    transforms.RandomCrop(CONFIG["img_size"], padding=12),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Val/Test — sadece resize + normalize
eval_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("Dönüşümler hazır.")
print(f"  Pre-train augmentation: 7 adım")
print(f"  Fine-tune augmentation: 7 adım")
print(f"  Eval: sadece Resize + Normalize")

In [ ]:
# ── FAZ 1: PlantCLEF DataLoader ───────────────────────────────────────────────
# PlantCLEF verisi ImageFolder formatında klasörlere ayrılmış olmalıdır.
# Eğer metadata CSV'si kullanılıyorsa aşağıdaki PlantCLEFDataset sınıfını kullanın.

plantclef_path = Path(CONFIG["plantclef_dir"])

if plantclef_path.exists():
    # ImageFolder yapısı varsa (train/ alt klasörüyle)
    if (plantclef_path / "train").exists():
        pretrain_dataset = datasets.ImageFolder(
            plantclef_path / "train", transform=pretrain_transforms
        )
    else:
        # Düz klasör yapısı — her tür adı bir alt klasör
        pretrain_dataset = datasets.ImageFolder(
            plantclef_path, transform=pretrain_transforms
        )
    pretrain_loader = DataLoader(
        pretrain_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=CONFIG["num_workers"],
        pin_memory=True,
    )
    actual_classes = len(pretrain_dataset.classes)
    CONFIG["plantclef_classes"] = actual_classes
    print(f"PlantCLEF yüklendi: {len(pretrain_dataset):,} görüntü | {actual_classes} tür")
else:
    print("PlantCLEF verisi henüz indirilmedi.")
    print(f"Beklenen dizin: {plantclef_path.resolve()}")
    print("Cell 1'deki wget komutlarını çalıştırın.")
    pretrain_loader = None

# ── FAZ 2: Fine-tune DataLoader (hastalık veri seti) ─────────────────────────
finetune_path = Path(CONFIG["finetune_dir"])
ft_train = datasets.ImageFolder(finetune_path / "train", transform=finetune_transforms)
ft_val   = datasets.ImageFolder(finetune_path / "val",   transform=eval_transforms)
ft_test  = datasets.ImageFolder(finetune_path / "test",  transform=eval_transforms)

ft_train_loader = DataLoader(ft_train, batch_size=CONFIG["batch_size"],
                             shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
ft_val_loader   = DataLoader(ft_val,   batch_size=CONFIG["batch_size"],
                             shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
ft_test_loader  = DataLoader(ft_test,  batch_size=CONFIG["batch_size"],
                             shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"\nFine-tune veri seti:")
print(f"  Train: {len(ft_train):,} | Val: {len(ft_val):,} | Test: {len(ft_test):,}")
print(f"  Sınıf sayısı: {len(ft_train.classes)}")

# Class weights (dengesizlik için)
ft_counts = np.array([len(list((finetune_path / "train" / c).glob("*.jpg")))
                      for c in ft_train.classes])
ft_weights = torch.tensor(
    len(ft_train) / (CONFIG["finetune_classes"] * ft_counts),
    dtype=torch.float32,
).to(DEVICE)

In [ ]:
# ── CNN Mimarisi — Botanix (Pre-train / Fine-tune desteği) ───────────────────
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))

class BotanixCNN(nn.Module):
    """
    Pre-train → Fine-tune desteği olan CNN.
    replace_head() ile classification head değiştirilebilir.
    freeze_backbone() / unfreeze_backbone() ile backbone dondurulabilir.
    """
    def __init__(self, num_classes=7806):
        super().__init__()
        # Backbone (pre-train ve fine-tune arasında paylaşılır)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),

            nn.Conv2d(64,  128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            ResidualBlock(128),

            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            ResidualBlock(256),

            nn.Conv2d(256, 512, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            ResidualBlock(512),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)

    def replace_head(self, num_classes):
        """Fine-tune için classification head'i yeni sınıf sayısıyla değiştirir."""
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.constant_(m.bias, 0)
        self.head = self.head.to(next(self.backbone.parameters()).device)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        x = self.backbone(x)
        x = self.gap(x)
        return self.head(x)

# FAZ 1 modeli — PlantCLEF sınıf sayısı ile
model = BotanixCNN(num_classes=CONFIG["plantclef_classes"]).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total_params:,}")
print(f"PlantCLEF sınıf sayısı: {CONFIG['plantclef_classes']}")

In [ ]:
# ── FAZ 1: Optimizer, Loss, Scheduler ────────────────────────────────────────
pretrain_criterion = nn.CrossEntropyLoss()  # PlantCLEF — balanced yeterince büyük dataset
pretrain_optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["pretrain_lr"],
    weight_decay=CONFIG["weight_decay"],
)

def warmup_cosine(epoch, warmup_epochs=5, total_epochs=30):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))

pretrain_scheduler = optim.lr_scheduler.LambdaLR(
    pretrain_optimizer,
    lr_lambda=lambda e: warmup_cosine(e, warmup_epochs=5, total_epochs=CONFIG["pretrain_epochs"])
)

print(f"Faz 1 Loss  : CrossEntropyLoss (ağırlıksız)")
print(f"Faz 1 LR    : {CONFIG['pretrain_lr']}")
print(f"Faz 1 Epoch : {CONFIG['pretrain_epochs']}")

In [ ]:
# ── Yardımcı Fonksiyonlar ─────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

# ── FAZ 1: PlantCLEF Pre-train ────────────────────────────────────────────────
if pretrain_loader is not None:
    print("=" * 60)
    print("FAZ 1 — PlantCLEF Pre-train Başlıyor")
    print("=" * 60)

    history_p1 = {"train_loss": [], "train_acc": [], "lr": []}
    train_start_p1 = time.time()

    for epoch in range(1, CONFIG["pretrain_epochs"] + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(
            model, pretrain_loader, pretrain_optimizer, pretrain_criterion, DEVICE
        )
        current_lr = pretrain_optimizer.param_groups[0]["lr"]
        pretrain_scheduler.step()

        history_p1["train_loss"].append(tr_loss)
        history_p1["train_acc"].append(tr_acc)
        history_p1["lr"].append(current_lr)

        print(f"[Pre-train] Epoch [{epoch:02d}/{CONFIG['pretrain_epochs']}] "
              f"Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
              f"LR: {current_lr:.2e} | {time.time()-t0:.1f}s")

    total_pretrain_time = time.time() - train_start_p1
    torch.save(model.state_dict(), CONFIG["pretrain_save"])
    print(f"\nPre-train tamamlandı: {total_pretrain_time/60:.1f} dakika")
    print(f"Model kaydedildi: {CONFIG['pretrain_save']}")
else:
    print("PlantCLEF verisi bulunamadı — Faz 1 atlandı.")
    print("Pre-train checkpoint varsa Faz 2'ye geçebilirsiniz.")

In [ ]:
# ── FAZ 2: Fine-tune — Classification Head Değiştir ──────────────────────────
print("=" * 60)
print("FAZ 2 — Fine-tune Başlıyor")
print("=" * 60)

# Pre-train checkpoint yükle (Faz 1 atlandıysa veya farklı çalıştırıldıysa)
pretrain_ckpt = Path(CONFIG["pretrain_save"])
if pretrain_ckpt.exists():
    model.load_state_dict(torch.load(pretrain_ckpt, map_location=DEVICE))
    print(f"Pre-train checkpoint yüklendi: {pretrain_ckpt}")
else:
    print("Pre-train checkpoint bulunamadı — ağırlıklar rastgele başlatıldı.")

# Head'i hastalık veri seti sınıf sayısıyla değiştir
model.replace_head(num_classes=CONFIG["finetune_classes"])
model = model.to(DEVICE)

# İlk N epoch backbone dondurulur
if CONFIG["freeze_backbone"]:
    model.freeze_backbone()
    print(f"Backbone donduruldu ({CONFIG['freeze_epochs']} epoch boyunca)")

ft_criterion = nn.CrossEntropyLoss(weight=ft_weights)
ft_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["finetune_lr"],
    weight_decay=CONFIG["weight_decay"],
)
ft_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=CONFIG["finetune_epochs"], eta_min=1e-6
)

history_ft = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_val_acc = 0.0
train_start_ft = time.time()

for epoch in range(1, CONFIG["finetune_epochs"] + 1):
    # Backbone açma zamanı geldi mi?
    if CONFIG["freeze_backbone"] and epoch == CONFIG["freeze_epochs"] + 1:
        model.unfreeze_backbone()
        ft_optimizer = optim.AdamW(
            model.parameters(),
            lr=CONFIG["finetune_lr"] * 0.1,   # backbone için daha düşük LR
            weight_decay=CONFIG["weight_decay"],
        )
        print(f"\nEpoch {epoch}: Backbone açıldı (LR={CONFIG['finetune_lr']*0.1:.2e})")

    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, ft_train_loader, ft_optimizer, ft_criterion, DEVICE)
    vl_loss, vl_acc = evaluate(model, ft_val_loader, ft_criterion, DEVICE)
    current_lr = ft_optimizer.param_groups[0]["lr"]
    ft_scheduler.step()

    history_ft["train_loss"].append(tr_loss)
    history_ft["train_acc"].append(tr_acc)
    history_ft["val_loss"].append(vl_loss)
    history_ft["val_acc"].append(vl_acc)
    history_ft["lr"].append(current_lr)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), CONFIG["finetune_save"])

    frozen = "❄" if (CONFIG["freeze_backbone"] and epoch <= CONFIG["freeze_epochs"]) else "🔥"
    print(f"[Fine-tune {frozen}] Epoch [{epoch:02d}/{CONFIG['finetune_epochs']}] "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | "
          f"LR: {current_lr:.2e} | {time.time()-t0:.1f}s")

total_finetune_time = time.time() - train_start_ft
print(f"\nFine-tune tamamlandı: {total_finetune_time/60:.1f} dakika")
print(f"En iyi Val Accuracy: {best_val_acc:.4f}")

In [ ]:
# ── Kapsamlı Grafiksel Değerlendirme ─────────────────────────────────────────
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import warnings; warnings.filterwarnings("ignore")

# ── Eğitim Grafikleri ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_ft["train_loss"], label="Train")
axes[0].plot(history_ft["val_loss"],   label="Val")
axes[0].axvline(x=CONFIG["freeze_epochs"]-1, color="gray", linestyle="--", label="Backbone açıldı")
axes[0].set_title("Fine-tune Loss"); axes[0].legend()
axes[1].plot(history_ft["train_acc"], label="Train")
axes[1].plot(history_ft["val_acc"],   label="Val")
axes[1].axvline(x=CONFIG["freeze_epochs"]-1, color="gray", linestyle="--", label="Backbone açıldı")
axes[1].set_title("Fine-tune Accuracy"); axes[1].legend()
plt.suptitle("Botanix CNN — Fine-tune Training History")
plt.tight_layout()
plt.savefig("./results/botanix_cnn_scratch_training.png", dpi=150)
plt.show()

# ── Test Değerlendirmesi ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CONFIG["finetune_save"], map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
infer_start = time.time()
with torch.no_grad():
    for imgs, labels in ft_test_loader:
        outputs = model(imgs.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

infer_time = time.time() - infer_start
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Precision:      {prec:.4f}")
print(f"Recall:         {rec:.4f}")
print(f"F1 Score:       {f1:.4f}")
print(f"Inference Time: {infer_time/len(ft_test)*1000:.2f} ms/image")

# ── Kapsamlı Grafiksel Değerlendirme ─────────────────────────────────────────
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import warnings; warnings.filterwarnings("ignore")

MODEL_LABEL  = "Botanix CNN (PlantCLEF Pre-train → Fine-tune)"
MODEL_PREFIX = "botanix_cnn_scratch"
class_names  = ft_train.classes

# ── 1. Per-Class F1 Score ─────────────────────────────────────────────────────
report_dict = classification_report(all_labels, all_preds,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
per_class_f1 = {k: v["f1-score"] for k, v in report_dict.items()
                if k not in ("accuracy", "macro avg", "weighted avg")}
sorted_f1 = sorted(per_class_f1.items(), key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
worst20 = sorted_f1[:20]
best20  = sorted_f1[-20:][::-1]
axes[0].barh([x[0][:25] for x in worst20], [x[1] for x in worst20], color="#EF5350")
axes[0].set_title(f"{MODEL_LABEL} — En Düşük F1 (20 Sınıf)")
axes[0].set_xlabel("F1 Score"); axes[0].set_xlim(0, 1)
axes[1].barh([x[0][:25] for x in best20],  [x[1] for x in best20],  color="#66BB6A")
axes[1].set_title(f"{MODEL_LABEL} — En Yüksek F1 (20 Sınıf)")
axes[1].set_xlabel("F1 Score"); axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_per_class_f1.png", dpi=150)
plt.show()

# ── 2. Top-10 Karıştırılan Sınıf Çiftleri ────────────────────────────────────
cm_full = confusion_matrix(all_labels, all_preds)
np.fill_diagonal(cm_full, 0)
flat_idx = np.argsort(cm_full.ravel())[::-1][:10]
rows_e, cols_e = np.unravel_index(flat_idx, cm_full.shape)

fig, ax = plt.subplots(figsize=(12, 5))
labels_top = [f"{class_names[r][:15]}→{class_names[c][:15]}" for r, c in zip(rows_e, cols_e)]
counts_top = [cm_full[r, c] for r, c in zip(rows_e, cols_e)]
bars = ax.barh(labels_top[::-1], counts_top[::-1], color="#5C6BC0")
for bar, val in zip(bars, counts_top[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_title(f"{MODEL_LABEL} — En Çok Karıştırılan 10 Sınıf Çifti")
ax.set_xlabel("Yanlış Sınıflandırma Sayısı")
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_top_errors.png", dpi=150)
plt.show()

# ── 3. Makro-Ortalama ROC Eğrisi ─────────────────────────────────────────────
all_probs_sc = []
model.eval()
with torch.no_grad():
    for imgs, _ in ft_test_loader:
        out = model(imgs.to(DEVICE))
        all_probs_sc.extend(torch.softmax(out, dim=1).cpu().numpy())
all_probs_sc = np.array(all_probs_sc)

labels_bin = label_binarize(all_labels, classes=list(range(CONFIG["finetune_classes"])))
fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
for i in range(CONFIG["num_classes"]):
    if labels_bin[:, i].sum() > 0:
        fpr_dict[i], tpr_dict[i], _ = roc_curve(labels_bin[:, i], all_probs_sc[:, i])
        roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

all_fpr  = np.unique(np.concatenate([fpr_dict[i] for i in roc_auc_dict]))
mean_tpr = np.zeros_like(all_fpr)
for i in roc_auc_dict:
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= len(roc_auc_dict)
macro_auc = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(all_fpr, mean_tpr, color="#1565C0", lw=2,
        label=f"Makro-Ortalama ROC (AUC = {macro_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL_LABEL} — ROC Eğrisi (Makro Ortalama)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_roc.png", dpi=150)
plt.show()

print(f"\nMakro-Ortalama ROC AUC: {macro_auc:.4f}")
print("Grafikler ./results/ dizinine kaydedildi.")

In [ ]:
# ── Sonuçları Kaydet ──────────────────────────────────────────────────────────
results = {
    "model": "Botanix CNN (PlantCLEF Pre-train → Fine-tune)",
    "accuracy":       round(acc, 4),
    "precision":      round(prec, 4),
    "recall":         round(rec, 4),
    "f1_score":       round(f1, 4),
    "roc_auc_macro":  round(macro_auc, 4),
    "pretrain_time_min":  round(total_pretrain_time / 60, 2) if pretrain_loader else 0,
    "finetune_time_min":  round(total_finetune_time / 60, 2),
    "inference_time_ms_per_image": round(infer_time / len(ft_test) * 1000, 4),
    "best_val_acc":   round(best_val_acc, 4),
    "total_params":   total_params,
    "config":         CONFIG,
}
with open("./results/botanix_cnn_scratch_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## Mobil Uyumluluk Notu — Botanix CNN (PlantCLEF Pre-train → Fine-tune)

| Özellik | Değer |
|---------|-------|
| Mimari | Custom ResidualCNN (sıfırdan) |
| Parametre sayısı | ~8M |
| Model boyutu (.pth) | ~32 MB |
| Inference süresi (GPU) | < 5 ms/görüntü |
| Mobil uygunluğu | ✅ Yüksek |

**Avantajlar:**
- En düşük parametre sayısı → mobil belleğe en uygun model
- PlantCLEF pre-train ile botanik alan bilgisi kazanılmış
- Fine-tune ile kendi veri setine adapte edilmiş
- TorchScript dönüşümü en basit bu mimaride

**Potansiyel kısıtlar:**
- Büyük Transformer modellerine kıyasla doğruluk düşük olabilir
- Scratch eğitim nedeniyle pretrained ViT/Swin'den geride kalabilir

**Mobil dönüşüm için önerilen araçlar:** TorchScript (`torch.jit.trace`), Core ML Tools (iOS), TFLite (Android via ONNX), bu model ailesi üretim için en uygun adaydır